# Emoji reactions to posts: dual scaling

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Dual Scaling** · a worked example from the StatNex teaching guide

**The story.** A brand's social-media team counted the emoji reactions (Like, Love, Haha, Wow, Sad, Angry) to 11 types of post over a year, 2,221 reactions in total. Dual scaling gives post types and emotions scores so that the two line up as closely as possible.

**The question.** Can we place post types and emotions on a common scale?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/dual_scaling.csv` (12 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Title,Post type,Like,Love,Haha,Wow,Sad,Angry,Total
Post type,Product launch,120.0,40.0,8.0,65.0,2.0,6.0,241.0
,Behind the scenes,85.0,52.0,20.0,18.0,1.0,2.0,178.0
,Customer story,70.0,88.0,6.0,12.0,15.0,1.0,192.0
,Meme,60.0,25.0,140.0,10.0,2.0,4.0,241.0
,Discount offer,150.0,35.0,5.0,30.0,1.0,10.0,231.0
,Price increase,20.0,2.0,12.0,15.0,30.0,110.0,189.0
,Charity event,55.0,120.0,2.0,20.0,10.0,2.0,209.0
,Tutorial,95.0,30.0,10.0,25.0,1.0,3.0,164.0
,Giveaway,130.0,60.0,15.0,22.0,3.0,8.0,238.0
,Service outage,10.0,1.0,20.0,8.0,45.0,90.0,174.0
,Holiday greeting,75.0,70.0,10.0,5.0,3.0,1.0,164.0
,Total,870.0,523.0,248.0,230.0,113.0,237.0,2221.0
""")

## Helper: totals

A step that no library does, written out. In the code download it is the file `totals.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# A cross tabulation of counts: one category of the row variable per row, one of the
# column variable per column.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Like", "Love", "Haha", "Wow", "Sad", "Angry"]
table = data.set_index("Post type")[columns].apply(pd.to_numeric, errors="coerce")
# A row or column named like a total (Total, Sum, or a Japanese word for a total, as listed in totals.py) is a margin, not data,
# so it is left out, as the app does. Spaces, underscores, case and full-width letters
# are ignored when names are compared.
# is_total: defined in the helper cell totals above (totals.py in the code download)

table = table.loc[[not is_total(row) for row in table.index],
                  [not is_total(column) for column in table.columns]]
# A row with a blank count is left out.
table = table.dropna()
counts = table.to_numpy(float)
n_rows, n_columns = counts.shape
row_labels, column_labels = list(table.index), list(table.columns)

## Cross tabulation

In [ ]:
crosstab = pd.concat([table, table.sum(axis=1).rename("Total")], axis=1)
crosstab = pd.concat([crosstab, crosstab.sum().to_frame("Total").T])
print(crosstab)

## Decomposition

In [ ]:
# Dual scaling gives every row category a score and every column category a score, chosen
# so that the correlation ratio eta^2 -- the share of the scores' variation that lies
# between categories -- is as large as possible; then the next best at right angles, and
# so on. The answer is the singular value decomposition correspondence analysis uses: of
# the standardised residuals (p_ij - r_i c_j) / sqrt(r_i c_j), with eta^2 the squared
# singular value. An R x C table has min(R, C) - 1 axes.
P = counts / counts.sum()
row_mass, column_mass = P.sum(axis=1), P.sum(axis=0)
residual = (P - np.outer(row_mass, column_mass)) / np.sqrt(np.outer(row_mass, column_mass))
U, singular, Vt = np.linalg.svd(residual, full_matrices=False)
keep = min(n_rows, n_columns) - 1
U, singular, V = U[:, :keep], singular[:keep], Vt[:keep].T
# An axis with no inertia at all is dropped.
live = singular > 1e-12 * max(1.0, singular[0])
U, singular, V = U[:, live], singular[live], V[:, live]
# A weight vector and its negative are the same solution, so its direction is a
# convention. The app's: flip each axis so its largest row-side component (in absolute
# value) is positive. When two components tie for the largest (two rows of equal
# weight, say) rounding decides, and another program may choose the other sign.
for k in range(len(singular)):
    if U[np.argmax(np.abs(U[:, k])), k] < 0:
        U[:, k], V[:, k] = -U[:, k], -V[:, k]
eta2 = singular**2
share = eta2 / eta2.sum()
axes = [f"Axis {k + 1}" for k in range(len(singular))]

## Solution and test for each axis

In [ ]:
# Bartlett's chi-square = -[n - (R + C + 1) / 2] ln(1 - eta^2) on R + C - 2k - 1 degrees
# of freedom for axis k. An eta^2 of exactly 1 is capped just below 1, so the logarithm
# stays finite.
n = counts.sum()
chi2 = -(n - (n_rows + n_columns + 1) / 2) * np.log1p(-np.minimum(eta2, 1 - 1e-12))
df = n_rows + n_columns - 2 * np.arange(1, len(eta2) + 1) - 1
solution = pd.DataFrame({"eta^2": eta2, "share": share, "cumulative share": share.cumsum(),
                         "chi-square": chi2, "df": df, "P": stats.chi2.sf(chi2, df)}, index=axes)
print()
print(solution)

## Optimal weight vectors

In [ ]:
# Unweighted: each category's raw score, the singular vector divided by sqrt(mass). Every
# axis has variance 1, so compare categories within an axis, not across axes.
# Weighted by the correlation ratio: the same scores times eta (the singular value), so an
# axis that explains little shrinks towards the origin. This is the one to plot.
plain = pd.DataFrame(np.vstack([U / np.sqrt(row_mass)[:, None], V / np.sqrt(column_mass)[:, None]]),
                     index=pd.MultiIndex.from_arrays([["Row"] * n_rows + ["Column"] * n_columns,
                                                      row_labels + column_labels]),
                     columns=axes)
scaled = plain * singular
print()
print(plain.round(4))
print()
print(scaled.round(4))

## What the result says

Dual scaling gives each post type and each emotion a weight on a scale, so that posts and the emotions they trigger get similar weights. The correlation ratio η² says how well a scale separates them. It is 0.554 for axis 1 and 0.282 for axis 2. Their shares are 56.6% and 28.8%, so two scales carry 85.3% of the pattern.

On axis 1, Price increase (2.20) and Service outage (2.29) get high weights with Angry (2.45) and Sad (1.80), and all other post types get negative weights. On axis 2, Meme (2.82) and Haha (2.80) stand alone.

So yes, a common scale exists. The first runs from calm everyday posts to anger, and the second picks out humour.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.